In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('ggplot')
pd.set_option('display.max_columns', 20)


c:\Users\COMPUTAEX\anaconda3\envs\aove\Lib\site-packages\xgboost\core.py:39: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.1)
  import scipy.sparse


In [3]:
filepath = '../Datasets/continuous_factory_process.csv'

raw = pd.read_csv(filepath, header=0, parse_dates=['time_stamp'], decimal='.')
raw = raw.set_index('time_stamp').sort_index()

print(raw.shape)
raw.head()


(14088, 115)


,AmbientConditions.AmbientHumidity.U.Actual,AmbientConditions.AmbientTemperature.U.Actual,Machine1.RawMaterial.Property1,Machine1.RawMaterial.Property2,Machine1.RawMaterial.Property3,Machine1.RawMaterial.Property4,Machine1.RawMaterialFeederParameter.U.Actual,Machine1.Zone1Temperature.C.Actual,Machine1.Zone2Temperature.C.Actual,Machine1.MotorAmperage.U.Actual,...,Stage2.Output.Measurement10.U.Actual,Stage2.Output.Measurement10.U.Setpoint,Stage2.Output.Measurement11.U.Actual,Stage2.Output.Measurement11.U.Setpoint,Stage2.Output.Measurement12.U.Actual,Stage2.Output.Measurement12.U.Setpoint,Stage2.Output.Measurement13.U.Actual,Stage2.Output.Measurement13.U.Setpoint,Stage2.Output.Measurement14.U.Actual,Stage2.Output.Measurement14.U.Setpoint
time_stamp,,,,,,,,,,,,,,,,,,,,,
2019-03-06 10:52:33,17.24,23.53,11.54,200,963.0,247,1241.26,72.0,72.3,48.03,...,0.0,7.93,0.0,5.65,0.0,1.85,0.0,2.89,0.0,11.71
2019-03-06 10:52:34,17.24,23.53,11.54,200,963.0,247,1246.09,72.0,72.3,48.03,...,0.0,7.93,0.0,5.65,0.0,1.85,0.0,2.89,0.0,11.71
2019-03-06 10:52:35,17.24,23.53,11.54,200,963.0,247,1246.29,72.0,72.3,48.16,...,0.0,7.93,0.0,5.65,0.0,1.85,0.0,2.89,0.0,11.71
2019-03-06 10:52:36,17.24,23.53,11.54,200,963.0,247,1247.59,72.0,72.3,48.57,...,0.0,7.93,0.0,5.65,0.0,1.85,0.0,2.89,0.0,11.71
2019-03-06 10:52:37,17.24,23.53,11.54,200,963.0,247,1252.83,72.1,72.4,48.57,...,0.0,7.93,0.0,5.65,0.0,1.85,0.0,2.89,0.0,11.71


# Check statistics and missing values 

# Ver forma, tipos y lista de columnas
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'min': df.min(numeric_only=True),
    'max': df.max(numeric_only=True),
    'mean': df.mean(numeric_only=True),
    'std': df.std(numeric_only=True),
    'pct_missing': df.isna().mean() * 100,
    'pct_zero': (df == 0).mean() * 100,
    'n_unique': df.nunique(),
})

# CV normaliza la desviación estándar sobre la media. Da una idea de si la
# variación es estable — PERO ojo: un CV bajo puede ser control fino real
# o un sensor congelado; no se puede decidir con este número aislado, ver
# más abajo el caso Air Flow 04/05.
summary['cv'] = summary['std'] / summary['mean'].abs()

summary